# 第12章：Agent 基础

> 🎯 理解Agent核心概念，构建能使用工具的智能体

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(
    base_url=os.getenv('LM_STUDIO_BASE_URL', 'http://localhost:1234/v1'),
    api_key='lm-studio',
    model=os.getenv('LM_STUDIO_MODEL', 'qwen2.5-7b-instruct')
)

## 1. Agent 核心概念

Agent = LLM + Tools + 决策循环

```
用户输入 → LLM思考 → 选择工具 → 执行 → 观察结果 → 继续/结束
```

## 2. 定义工具

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    '''计算数学表达式，如：2+3*4'''
    try:
        return str(eval(expression))
    except:
        return '计算错误'

@tool
def get_weather(city: str) -> str:
    '''获取城市天气'''
    weather = {'北京': '晴天 25°C', '上海': '多云 28°C', '广州': '小雨 30°C'}
    return weather.get(city, f'{city}天气未知')

@tool
def search(query: str) -> str:
    '''搜索信息'''
    return f'搜索结果：关于"{query}"的相关信息...'

tools = [calculator, get_weather, search]
print('工具列表:')
for t in tools:
    print(f'- {t.name}: {t.description}')

## 3. 创建 Agent

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个有帮助的助手，可以使用工具来回答问题。'),
    ('placeholder', '{chat_history}'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}')
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 4. 测试 Agent

In [ ]:
# 测试计算
result = agent_executor.invoke({'input': '计算 123 * 456 的结果'})
print('\n最终答案:', result['output'])

In [ ]:
# 测试天气
result = agent_executor.invoke({'input': '北京天气怎么样？'})
print('\n最终答案:', result['output'])

In [ ]:
# 测试搜索
result = agent_executor.invoke({'input': '搜索一下Python的最新版本'})
print('\n最终答案:', result['output'])

## 5. ReAct 思考过程

Agent的思考模式：
```
Thought: 我需要...
Action: 调用工具
Observation: 工具返回结果
Thought: 根据结果...
Final Answer: 最终回答
```

## 📝 小结
1. **Agent组成**：LLM + Tools + Prompt
2. **工具定义**：`@tool` 装饰器
3. **执行器**：`AgentExecutor`
4. **ReAct**：思考-行动-观察循环